# Pipeline vs Baseline Experiment

In [ ]:
import sys
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

BASELINE_DIR = REPO_ROOT / "baseline"
if str(BASELINE_DIR) not in sys.path:
    sys.path.insert(0, str(BASELINE_DIR))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = REPO_ROOT / "database" / "training_data.json"
EXPERIMENT_DIR = REPO_ROOT / "experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
import logging
from datetime import datetime

from logger import configure_logging

TRACE_LOG_PATH = EXPERIMENT_DIR / f"experimentation_pipeline_vs_baseline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
configure_logging(
    level="INFO",
    log_file=str(TRACE_LOG_PATH),
    enable_console=True,
    enable_file=True,
)

exp_logger = logging.getLogger("experimentation_pipeline_vs_baseline")
exp_logger.info("Trace logging initialized")
print(f"Trace log file: {TRACE_LOG_PATH}")

In [ ]:
TARGET = {
    "dbaasp_id": "DBAASPS_373",
    "sequence": "KLFKRWKHLFR",
    "length": 11,
    "smiles": "CC(C)C[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CCCCN)NC(=O)[C@H](Cc1ccccc1)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](N)CCCCN)C(=O)N[C@@H](Cc1ccccc1)C(=O)N[C@@H](CCCN=C(N)N)C(=O)O",
    "ph": None,
    "molecular_weight": 1558.9480000000003,
    "logp": -0.992100000000006,
    "net_charge": 5.0,
    "isoelectric_point": 12.18,
    "hydrophobicity": 1.05,
    "cathionicity": 6,
    "target_groups": ["Gram+"],
    "complexity": "Monomer"
}

SHARED_MAX_LEN = 14
SHARED_BATCH_SIZE = 32
SHARED_EPOCHS = 100
SHARED_LATENT_DIM = 32

# Tuned pipeline hyperparameters (from pipeline_experimentations reference setup).
PIPELINE_LATENT_DIM = 64
PIPELINE_HIDDEN_DIM = 512
PIPELINE_EPOCHS = 120
PIPELINE_BATCH_SIZE = 64
PIPELINE_LR = 0.05
PIPELINE_KL_ANNEAL_EPOCHS = 40

NB_ITERATIONS = 50
NB_PEPTIDES = 300
TOP_K = 10
BASELINE_NUM_SAMPLES = 10

print(TARGET)
print(
    f"Pipeline tuned config -> latent={PIPELINE_LATENT_DIM}, hidden={PIPELINE_HIDDEN_DIM}, "
    f"epochs={PIPELINE_EPOCHS}, batch={PIPELINE_BATCH_SIZE}, lr={PIPELINE_LR}, kl_anneal={PIPELINE_KL_ANNEAL_EPOCHS}"
)

In [ ]:
from training import train_model
from inference import generate_peptides

baseline_model_path = EXPERIMENT_DIR / "baseline_cvae_model.pth"
baseline_scaler_path = EXPERIMENT_DIR / "baseline_scaler.pkl"

baseline_model, _ = train_model(
    dataset_file=str(DATA_PATH),
    scaler_path=str(baseline_scaler_path),
    batch_size=SHARED_BATCH_SIZE,
    max_len=SHARED_MAX_LEN,
    epochs=SHARED_EPOCHS,
    latent_dim=SHARED_LATENT_DIM,
    model_path=str(baseline_model_path),
)

with open(baseline_scaler_path, "rb") as f:
    baseline_scaler = pickle.load(f)

baseline_target = [
    TARGET["length"],
    7.0 if TARGET["ph"] is None else float(TARGET["ph"]),
    TARGET["molecular_weight"],
    TARGET["logp"],
    TARGET["net_charge"],
    TARGET["isoelectric_point"],
    TARGET["hydrophobicity"],
    TARGET["cathionicity"],
]

baseline_sequences = generate_peptides(
    model=baseline_model,
    scaler=baseline_scaler,
    num_samples=BASELINE_NUM_SAMPLES,
    properties=baseline_target,
    temperature=0.9,
    top_k=5,
)

In [ ]:
from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader
from peptide_pipeline.generator.cvae_generator_agent.cvae_generator import CVAEGenerator

AA = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX = {aa: i for i, aa in enumerate(AA)}
PAD_IDX = 20
VOCAB_SIZE = 21


def encode_one_hot_with_pad(sequences, max_len):
    x = torch.zeros(len(sequences), max_len * VOCAB_SIZE, dtype=torch.float32)
    for i, seq in enumerate(sequences):
        for pos in range(max_len):
            x[i, pos * VOCAB_SIZE + PAD_IDX] = 1.0
        for pos, aa in enumerate(seq[:max_len]):
            if aa in AA_TO_IDX:
                x[i, pos * VOCAB_SIZE + PAD_IDX] = 0.0
                x[i, pos * VOCAB_SIZE + AA_TO_IDX[aa]] = 1.0
    return x


def build_condition_tensor(dataframe, model):
    cond = torch.zeros(len(dataframe), model.condition_dim, dtype=torch.float32)
    cond[:, 0] = torch.tensor(dataframe["length"].values, dtype=torch.float32)
    cond[:, 1] = torch.tensor(dataframe["molecular_weight"].values, dtype=torch.float32)
    cond[:, 2] = torch.tensor(dataframe["net_charge"].values, dtype=torch.float32)
    cond[:, 3] = torch.tensor(dataframe["isoelectric_point"].values, dtype=torch.float32)
    cond[:, 4] = torch.tensor(dataframe["hydrophobicity"].values, dtype=torch.float32)
    cond[:, 5] = torch.tensor(dataframe["cathionicity"].values, dtype=torch.float32)
    cond[:, 6] = 0.5
    cond[:, 7] = torch.tensor(dataframe["logp"].values, dtype=torch.float32)
    cond[:, 8] = 0.0
    cond[:, 9] = 5.0
    cond[:, 10] = 5.0
    cond[:, 11] = 100.0
    return cond


loader = JSONDataLoader()
loader.load_data(
    source=str(DATA_PATH),
    columns=[
        "sequence",
        "length",
        "ph",
        "molecular_weight",
        "logp",
        "net_charge",
        "isoelectric_point",
        "hydrophobicity",
        "cathionicity",
    ],
    fillna_defaults={
        "length": 10,
        "ph": 7.0,
        "molecular_weight": 1500.0,
        "logp": 0.0,
        "net_charge": 0.0,
        "isoelectric_point": 7.0,
        "hydrophobicity": 0.0,
        "cathionicity": 0.0,
    },
    normalize_sequence=True,
    sequence_column="sequence",
    keep_standard_amino_acids_only=True,
)

pipeline_df = loader.get_data().copy()

pipeline_cvae = CVAEGenerator(
    max_len=SHARED_MAX_LEN,
    latent_dim=PIPELINE_LATENT_DIM,
    hidden_dim=PIPELINE_HIDDEN_DIM,
    condition_dim=32,
)

pipeline_model_path = EXPERIMENT_DIR / (
    f"pipeline_cvae_model_lat{PIPELINE_LATENT_DIM}_hid{PIPELINE_HIDDEN_DIM}_ep{PIPELINE_EPOCHS}_bs{PIPELINE_BATCH_SIZE}.pth"
)

x = encode_one_hot_with_pad(pipeline_df["sequence"].tolist(), max_len=SHARED_MAX_LEN)
conditions = build_condition_tensor(pipeline_df, pipeline_cvae)
lengths = torch.tensor(pipeline_df["length"].astype(int).values, dtype=torch.long)

x = x.to(pipeline_cvae.device)
conditions = conditions.to(pipeline_cvae.device)
lengths = lengths.to(pipeline_cvae.device)

if pipeline_model_path.exists():
    pipeline_cvae.load_model(str(pipeline_model_path))
else:
    pipeline_cvae.train_model(
        data=x,
        conditions=conditions,
        lengths=lengths,
        epochs=PIPELINE_EPOCHS,
        batch_size=PIPELINE_BATCH_SIZE,
        lr=PIPELINE_LR,
        kl_anneal_epochs=PIPELINE_KL_ANNEAL_EPOCHS,
    )
    pipeline_cvae.save_model(str(pipeline_model_path))

print(f"Using pipeline model path: {pipeline_model_path.name}")

In [ ]:
from peptide_pipeline.chemist.chemist_agent.config_chemist import ChemistConfig, RangeTarget
from peptide_pipeline.chemist.chemist_agent.chemist_agent import ChemistAgent
from peptide_pipeline.orchestrator.orchestrator_agent.orchestrator import Orchestrator
from peptide_pipeline.biologist.esm_l2_bio_agent.esm_biologist_global_l2 import ESMBiologistGlobalL2
from peptide_pipeline.biologist.base import BaseBiologist


class FallbackBiologist(BaseBiologist):
    def __init__(self, reference_peptide):
        self.reference = reference_peptide

    def score_peptides(self, peptides):
        scores = []
        ref_set = set(self.reference)
        for p in peptides:
            common = len(ref_set.intersection(set(p)))
            scores.append(common / max(len(set(self.reference)), 1))
        return scores


chemist_config = ChemistConfig(
    ph=7.0 if TARGET["ph"] is None else float(TARGET["ph"]),
    length=RangeTarget(min=8.0, max=14.0, target=float(TARGET["length"]), weight=1.0),
    molecular_weight=RangeTarget(min=1200.0, max=2000.0, target=float(TARGET["molecular_weight"]), weight=1.0),
    logp=RangeTarget(min=-3.0, max=3.0, target=float(TARGET["logp"]), weight=1.0),
    net_charge=RangeTarget(min=2.0, max=8.0, target=float(TARGET["net_charge"]), weight=1.0),
    isoelectric_point=RangeTarget(min=9.0, max=14.0, target=float(TARGET["isoelectric_point"]), weight=1.0),
    hydrophobicity=RangeTarget(min=-2.0, max=3.0, target=float(TARGET["hydrophobicity"]), weight=1.0),
)

chemist = ChemistAgent(config=chemist_config)

try:
    biologist = ESMBiologistGlobalL2(
        reference_peptide=TARGET["sequence"],
        batch_size=16,
        score_temperature=50.0,
    )
except Exception as e:
    print(f"Falling back to lightweight biologist: {e}")
    biologist = FallbackBiologist(reference_peptide=TARGET["sequence"])

orchestrator = Orchestrator(generator=pipeline_cvae, chemist=chemist, biologist=biologist)
pipeline_results = orchestrator.run(
    nb_iterations=NB_ITERATIONS,
    nb_peptides=NB_PEPTIDES,
    top_k=TOP_K,
    exploration_rate=0.15,
    initial_peptide=TARGET["sequence"],
    final_target={
        "length": TARGET["length"],
        "molecular_weight": TARGET["molecular_weight"],
        "logp": TARGET["logp"],
        "net_charge": TARGET["net_charge"],
        "isoelectric_point": TARGET["isoelectric_point"],
        "hydrophobicity": TARGET["hydrophobicity"],
        "cathionicity": TARGET["cathionicity"],
    },
)

In [ ]:
import csv

pipeline_display = []
for row in pipeline_results:
    item = {
        "sequence": row.get("peptide", ""),
        "score": row.get("score"),
        "chemist_score": row.get("chemist_score"),
        "biologist_score": row.get("biologist_score"),
        "length": row.get("properties", {}).get("length"),
        "molecular_weight": row.get("properties", {}).get("molecular_weight"),
        "net_charge": row.get("properties", {}).get("net_charge"),
        "isoelectric_point": row.get("properties", {}).get("isoelectric_point"),
        "hydrophobicity": row.get("properties", {}).get("hydrophobicity"),
        "logp": row.get("properties", {}).get("logp"),
        "in_limits": row.get("in_limits", False),
    }
    pipeline_display.append(item)

pipeline_display_df = pd.DataFrame(pipeline_display)
if not pipeline_display_df.empty:
    pipeline_display_df = pipeline_display_df.sort_values(by=["in_limits", "score"], ascending=[False, False]).reset_index(drop=True)
    for col in ["score", "chemist_score", "biologist_score", "molecular_weight", "net_charge", "isoelectric_point", "hydrophobicity", "logp"]:
        if col in pipeline_display_df.columns:
            pipeline_display_df[col] = pipeline_display_df[col].astype(float).round(3)

baseline_clean = [seq for seq in baseline_sequences if isinstance(seq, str) and seq]
baseline_df = pd.DataFrame({
    "rank": list(range(1, len(baseline_clean) + 1)),
    "sequence": baseline_clean,
})

target_df = pd.DataFrame([TARGET])
combined_csv_path = EXPERIMENT_DIR / "pipeline_vs_baseline_results.csv"

target_cols = list(target_df.columns)
pipeline_cols = [
    "sequence",
    "score",
    "chemist_score",
    "biologist_score",
    "in_limits",
    "length",
    "molecular_weight",
    "net_charge",
    "isoelectric_point",
    "hydrophobicity",
    "logp",
]
baseline_cols = ["rank", "sequence"]

with open(combined_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow(["TARGET"])
    writer.writerow(target_cols)
    writer.writerow([TARGET.get(col, "") for col in target_cols])
    writer.writerow([])

    writer.writerow(["PIPELINE_RESULTS"])
    writer.writerow(pipeline_cols)
    if not pipeline_display_df.empty:
        for _, row in pipeline_display_df.iterrows():
            writer.writerow([row.get(col, "") for col in pipeline_cols])
    writer.writerow([])

    writer.writerow(["BASELINE_RESULTS"])
    writer.writerow(baseline_cols)
    if not baseline_df.empty:
        for _, row in baseline_df.iterrows():
            writer.writerow([row.get(col, "") for col in baseline_cols])

In [ ]:
print("Pipeline results:")
if pipeline_display_df.empty:
    print("No peptide returned by the pipeline.")
else:
    print(f"Total peptides: {len(pipeline_display_df)}")
    display(pipeline_display_df)

print("Basline results:")
if baseline_df.empty:
    print("No sequence returned by the baseline.")
else:
    print(f"Total sequences: {len(baseline_df)}")
    for i, seq in enumerate(baseline_df["sequence"].tolist(), start=1):
        print(f"{i:02d}. {seq}")

## Simple Scaling Benchmark (Baseline vs Pipeline)

This section adds a small and practical scaling comparison between baseline and pipeline.

What is compared:
- Model size (latent dimension for baseline, latent+hidden dimensions for pipeline)
- Number of epochs
- Runtime (seconds)
- Agent-based quality metrics computed with the same evaluators for both methods:
  - Chemist score
  - Biologist score
  - Combined score = 0.5 * (chemist + biologist)
  - In-limits rate from chemist constraints

Notes:
- The benchmark is intentionally lightweight so it can run in a notebook.
- We still compute identity to target as a secondary reference, but the main comparison uses chemist/biologist scores.

In [ ]:
import time
import pickle

import matplotlib.pyplot as plt

from baseline.training import train_model
from baseline.inference import generate_peptides

from peptide_pipeline.dataloader.dataloader_json import DataLoader as JSONDataLoader
from peptide_pipeline.generator.cvae_generator_agent.cvae_generator import CVAEGenerator
from peptide_pipeline.chemist.chemist_agent.config_chemist import ChemistConfig, RangeTarget
from peptide_pipeline.chemist.chemist_agent.chemist_agent import ChemistAgent
from peptide_pipeline.orchestrator.orchestrator_agent.orchestrator import Orchestrator
from peptide_pipeline.biologist.base import BaseBiologist


def sequence_identity(seq, ref):
    if not seq or not ref:
        return 0.0
    n = min(len(seq), len(ref))
    matches = sum(1 for a, b in zip(seq[:n], ref[:n]) if a == b)
    return matches / max(len(ref), 1)


def summarize_identity(sequences, ref):
    clean = [s for s in sequences if isinstance(s, str) and s]
    if not clean:
        return {"best_identity": 0.0, "mean_identity": 0.0, "count": 0}
    ids = [sequence_identity(s, ref) for s in clean]
    return {
        "best_identity": float(max(ids)),
        "mean_identity": float(sum(ids) / len(ids)),
        "count": len(clean),
    }


def summarize_agent_scores(scored_rows):
    if not scored_rows:
        return {
            "chemist_mean": 0.0,
            "biologist_mean": 0.0,
            "combined_mean": 0.0,
            "combined_best": 0.0,
            "in_limits_rate": 0.0,
            "scored_count": 0,
        }

    chem = [float(r.get("chemist_score", 0.0)) for r in scored_rows]
    bio = [float(r.get("biologist_score", 0.0)) for r in scored_rows]
    comb = [float(r.get("combined_score", 0.0)) for r in scored_rows]
    in_limits = [1.0 if bool(r.get("in_limits", False)) else 0.0 for r in scored_rows]

    return {
        "chemist_mean": float(np.mean(chem)),
        "biologist_mean": float(np.mean(bio)),
        "combined_mean": float(np.mean(comb)),
        "combined_best": float(np.max(comb)),
        "in_limits_rate": float(np.mean(in_limits)),
        "scored_count": int(len(scored_rows)),
    }


def score_sequences_with_agents(sequences, chemist, biologist):
    clean = [s for s in sequences if isinstance(s, str) and s]
    if not clean:
        return []

    chem_rows = chemist.evaluate_peptides(clean)
    valid_rows = [r for r in chem_rows if isinstance(r, dict) and r.get("sequence")]
    peptides = [r["sequence"] for r in valid_rows]

    if not peptides:
        return []

    bio_scores = biologist.score_peptides(peptides)

    rows = []
    for c_row, b_score in zip(valid_rows, bio_scores):
        chem_score = float(c_row.get("score", 0.0))
        bio_score = float(b_score)
        combined = 0.5 * (chem_score + bio_score)
        rows.append(
            {
                "peptide": c_row["sequence"],
                "chemist_score": chem_score,
                "biologist_score": bio_score,
                "combined_score": combined,
                "in_limits": bool(c_row.get("in_limits", False)),
            }
        )
    return rows


class LocalFallbackBiologist(BaseBiologist):
    def __init__(self, reference_peptide):
        self.reference = reference_peptide

    def score_peptides(self, peptides):
        return [sequence_identity(p, self.reference) for p in peptides]


    def predict_activity(self, peptides, context=None):
        return self.score_peptides(peptides)


def get_biologist(reference_peptide):
    try:
        from peptide_pipeline.biologist.esm_l2_bio_agent.esm_biologist_global_l2 import ESMBiologistGlobalL2

        return ESMBiologistGlobalL2(reference_peptide=reference_peptide, batch_size=16, score_temperature=50.0)
    except Exception as e:
        print(f"Using fallback biologist for benchmark: {e}")
        return LocalFallbackBiologist(reference_peptide=reference_peptide)


def build_chemist_from_target(target):
    return ChemistAgent(
        config=ChemistConfig(
            ph=7.0 if target["ph"] is None else float(target["ph"]),
            length=RangeTarget(min=8.0, max=14.0, target=float(target["length"]), weight=1.0),
            molecular_weight=RangeTarget(min=1200.0, max=2000.0, target=float(target["molecular_weight"]), weight=1.0),
            logp=RangeTarget(min=-3.0, max=3.0, target=float(target["logp"]), weight=1.0),
            net_charge=RangeTarget(min=2.0, max=8.0, target=float(target["net_charge"]), weight=1.0),
            isoelectric_point=RangeTarget(min=9.0, max=14.0, target=float(target["isoelectric_point"]), weight=1.0),
            hydrophobicity=RangeTarget(min=-2.0, max=3.0, target=float(target["hydrophobicity"]), weight=1.0),
        )
    )


def get_pipeline_training_tensors(max_len):
    if all(k in globals() for k in ["pipeline_df", "x", "conditions", "lengths"]):
        return pipeline_df.copy(), x, conditions, lengths

    aa = "ACDEFGHIKLMNPQRSTVWY"
    aa_to_idx = {a: i for i, a in enumerate(aa)}
    pad_idx = 20
    vocab_size = 21

    def encode_one_hot_with_pad_local(sequences, max_len_local):
        out = torch.zeros(len(sequences), max_len_local * vocab_size, dtype=torch.float32)
        for i, seq in enumerate(sequences):
            for pos in range(max_len_local):
                out[i, pos * vocab_size + pad_idx] = 1.0
            for pos, ch in enumerate(seq[:max_len_local]):
                if ch in aa_to_idx:
                    out[i, pos * vocab_size + pad_idx] = 0.0
                    out[i, pos * vocab_size + aa_to_idx[ch]] = 1.0
        return out

    def build_condition_tensor_local(dataframe, model):
        cond_local = torch.zeros(len(dataframe), model.condition_dim, dtype=torch.float32)
        cond_local[:, 0] = torch.tensor(dataframe["length"].values, dtype=torch.float32)
        cond_local[:, 1] = torch.tensor(dataframe["molecular_weight"].values, dtype=torch.float32)
        cond_local[:, 2] = torch.tensor(dataframe["net_charge"].values, dtype=torch.float32)
        cond_local[:, 3] = torch.tensor(dataframe["isoelectric_point"].values, dtype=torch.float32)
        cond_local[:, 4] = torch.tensor(dataframe["hydrophobicity"].values, dtype=torch.float32)
        cond_local[:, 5] = torch.tensor(dataframe["cathionicity"].values, dtype=torch.float32)
        cond_local[:, 6] = 0.5
        cond_local[:, 7] = torch.tensor(dataframe["logp"].values, dtype=torch.float32)
        cond_local[:, 8] = 0.0
        cond_local[:, 9] = 5.0
        cond_local[:, 10] = 5.0
        cond_local[:, 11] = 100.0
        return cond_local

    local_loader = JSONDataLoader()
    local_loader.load_data(
        source=str(DATA_PATH),
        columns=[
            "sequence",
            "length",
            "ph",
            "molecular_weight",
            "logp",
            "net_charge",
            "isoelectric_point",
            "hydrophobicity",
            "cathionicity",
        ],
        fillna_defaults={
            "length": 10,
            "ph": 7.0,
            "molecular_weight": 1500.0,
            "logp": 0.0,
            "net_charge": 0.0,
            "isoelectric_point": 7.0,
            "hydrophobicity": 0.0,
            "cathionicity": 0.0,
        },
        normalize_sequence=True,
        sequence_column="sequence",
        keep_standard_amino_acids_only=True,
    )
    df_local = local_loader.get_data().copy()

    cvae_tmp = CVAEGenerator(max_len=max_len, latent_dim=16, hidden_dim=128, condition_dim=32)
    x_local = encode_one_hot_with_pad_local(df_local["sequence"].tolist(), max_len_local=max_len)
    cond_local = build_condition_tensor_local(df_local, cvae_tmp)
    lengths_local = torch.tensor(df_local["length"].astype(int).values, dtype=torch.long)

    return df_local, x_local, cond_local, lengths_local


benchmark_settings = [
    {"label": "small", "epochs": 10, "latent_dim": 16, "hidden_dim": 128},
    {"label": "larger", "epochs": 20, "latent_dim": 32, "hidden_dim": 256},
]

bench_iterations = 8
bench_peptides = 48
bench_top_k = 10
bench_baseline_samples = 48
benchmark_repeats = 3

benchmark_rows = []
ref_sequence = TARGET["sequence"]

pipe_df_local, x_local, cond_local, lengths_local = get_pipeline_training_tensors(SHARED_MAX_LEN)
x_local = x_local.to(torch.float32)
cond_local = cond_local.to(torch.float32)
lengths_local = lengths_local.to(torch.long)

benchmark_biologist = get_biologist(ref_sequence)

for cfg in benchmark_settings:
    label = cfg["label"]
    epochs = int(cfg["epochs"])
    latent_dim = int(cfg["latent_dim"])
    hidden_dim = int(cfg["hidden_dim"])

    for rep in range(1, benchmark_repeats + 1):
        seed_val = SEED + 50_000 + (1000 * (1 if label == "small" else 2)) + rep
        random.seed(seed_val)
        np.random.seed(seed_val)
        torch.manual_seed(seed_val)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed_val)

        chemist_cfg_agent = build_chemist_from_target(TARGET)

        baseline_model_path_cfg = EXPERIMENT_DIR / f"baseline_scale_{label}_rep{rep}_lat{latent_dim}_ep{epochs}.pth"
        baseline_scaler_path_cfg = EXPERIMENT_DIR / f"baseline_scale_{label}_rep{rep}_lat{latent_dim}_ep{epochs}_scaler.pkl"

        t0 = time.perf_counter()
        baseline_model_cfg, _ = train_model(
            dataset_file=str(DATA_PATH),
            scaler_path=str(baseline_scaler_path_cfg),
            batch_size=SHARED_BATCH_SIZE,
            max_len=SHARED_MAX_LEN,
            epochs=epochs,
            latent_dim=latent_dim,
            model_path=str(baseline_model_path_cfg),
        )

        with open(baseline_scaler_path_cfg, "rb") as f:
            baseline_scaler_cfg = pickle.load(f)

        baseline_target_cfg = [
            TARGET["length"],
            7.0 if TARGET["ph"] is None else float(TARGET["ph"]),
            TARGET["molecular_weight"],
            TARGET["logp"],
            TARGET["net_charge"],
            TARGET["isoelectric_point"],
            TARGET["hydrophobicity"],
            TARGET["cathionicity"],
        ]
        baseline_sequences_cfg = generate_peptides(
            model=baseline_model_cfg,
            scaler=baseline_scaler_cfg,
            num_samples=bench_baseline_samples,
            properties=baseline_target_cfg,
            temperature=0.9,
            top_k=5,
        )
        baseline_time = float(time.perf_counter() - t0)

        baseline_identity = summarize_identity(baseline_sequences_cfg, ref_sequence)
        baseline_scored_rows = score_sequences_with_agents(baseline_sequences_cfg, chemist_cfg_agent, benchmark_biologist)
        baseline_agent_metrics = summarize_agent_scores(baseline_scored_rows)

        benchmark_rows.append(
            {
                "method": "baseline",
                "setting": label,
                "repeat": rep,
                "epochs": epochs,
                "latent_dim": latent_dim,
                "hidden_dim": None,
                "runtime_sec": baseline_time,
                **baseline_identity,
                **baseline_agent_metrics,
            }
        )

        pipeline_model_path_cfg = EXPERIMENT_DIR / f"pipeline_scale_{label}_rep{rep}_lat{latent_dim}_hid{hidden_dim}_ep{epochs}.pth"

        t1 = time.perf_counter()
        pipeline_cvae_cfg = CVAEGenerator(
            max_len=SHARED_MAX_LEN,
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            condition_dim=32,
        )

        pipeline_cvae_cfg.train_model(
            data=x_local.to(pipeline_cvae_cfg.device),
            conditions=cond_local.to(pipeline_cvae_cfg.device),
            lengths=lengths_local.to(pipeline_cvae_cfg.device),
            epochs=epochs,
            batch_size=SHARED_BATCH_SIZE,
            lr=1e-3,
            kl_anneal_epochs=max(1, epochs // 2),
        )
        pipeline_cvae_cfg.save_model(str(pipeline_model_path_cfg))

        orchestrator_cfg = Orchestrator(
            generator=pipeline_cvae_cfg,
            chemist=chemist_cfg_agent,
            biologist=benchmark_biologist,
        )

        pipe_rows_cfg = orchestrator_cfg.run(
            nb_iterations=bench_iterations,
            nb_peptides=bench_peptides,
            top_k=bench_top_k,
            exploration_rate=0.15,
            initial_peptide=ref_sequence,
            final_target={
                "length": TARGET["length"],
                "molecular_weight": TARGET["molecular_weight"],
                "logp": TARGET["logp"],
                "net_charge": TARGET["net_charge"],
                "isoelectric_point": TARGET["isoelectric_point"],
                "hydrophobicity": TARGET["hydrophobicity"],
                "cathionicity": TARGET["cathionicity"],
            },
        )
        pipeline_time = float(time.perf_counter() - t1)

        pipe_sequences_cfg = [r.get("peptide", "") for r in pipe_rows_cfg if isinstance(r, dict)]
        pipeline_identity = summarize_identity(pipe_sequences_cfg, ref_sequence)
        pipeline_scored_rows = [
            {
                "peptide": r.get("peptide", ""),
                "chemist_score": float(r.get("chemist_score", 0.0)),
                "biologist_score": float(r.get("biologist_score", 0.0)),
                "combined_score": float(r.get("combined_score", r.get("score", 0.0))),
                "in_limits": bool(r.get("in_limits", False)),
            }
            for r in pipe_rows_cfg
            if isinstance(r, dict)
        ]
        pipeline_agent_metrics = summarize_agent_scores(pipeline_scored_rows)

        benchmark_rows.append(
            {
                "method": "pipeline",
                "setting": label,
                "repeat": rep,
                "epochs": epochs,
                "latent_dim": latent_dim,
                "hidden_dim": hidden_dim,
                "runtime_sec": pipeline_time,
                **pipeline_identity,
                **pipeline_agent_metrics,
            }
        )

benchmark_raw_df = pd.DataFrame(benchmark_rows)
benchmark_summary_df = pd.DataFrame()

if not benchmark_raw_df.empty:
    fixed_df = benchmark_raw_df.copy()
    fixed_df["hidden_dim_group"] = fixed_df["hidden_dim"].fillna(-1).astype(int)

    benchmark_summary_df = fixed_df.groupby(
        ["setting", "method", "epochs", "latent_dim", "hidden_dim_group"],
        as_index=False,
    ).agg(
        runtime_sec_mean=("runtime_sec", "mean"),
        runtime_sec_std=("runtime_sec", "std"),
        chemist_mean_mean=("chemist_mean", "mean"),
        chemist_mean_std=("chemist_mean", "std"),
        biologist_mean_mean=("biologist_mean", "mean"),
        biologist_mean_std=("biologist_mean", "std"),
        combined_mean_mean=("combined_mean", "mean"),
        combined_mean_std=("combined_mean", "std"),
        combined_best_mean=("combined_best", "mean"),
        combined_best_std=("combined_best", "std"),
        in_limits_rate_mean=("in_limits_rate", "mean"),
        in_limits_rate_std=("in_limits_rate", "std"),
        best_identity_mean=("best_identity", "mean"),
        best_identity_std=("best_identity", "std"),
        mean_identity_mean=("mean_identity", "mean"),
        mean_identity_std=("mean_identity", "std"),
        scored_count_mean=("scored_count", "mean"),
    )

    benchmark_summary_df["hidden_dim"] = benchmark_summary_df["hidden_dim_group"].replace({-1: np.nan})
    benchmark_summary_df = benchmark_summary_df.drop(columns=["hidden_dim_group"])
    benchmark_summary_df = benchmark_summary_df.sort_values(["setting", "method"]).reset_index(drop=True)

display(benchmark_summary_df)
display(benchmark_raw_df.head(8))

if not benchmark_summary_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)

    runtime_pivot = benchmark_summary_df.pivot(index="setting", columns="method", values="runtime_sec_mean")
    runtime_err_pivot = benchmark_summary_df.pivot(index="setting", columns="method", values="runtime_sec_std").fillna(0.0)
    runtime_pivot.plot(kind="bar", yerr=runtime_err_pivot, capsize=4, ax=axes[0], rot=0)
    axes[0].set_title(f"Runtime (mean +/- std, n={benchmark_repeats})")
    axes[0].set_xlabel("Setting")
    axes[0].set_ylabel("Seconds")

    combined_pivot = benchmark_summary_df.pivot(index="setting", columns="method", values="combined_mean_mean")
    combined_err_pivot = benchmark_summary_df.pivot(index="setting", columns="method", values="combined_mean_std").fillna(0.0)
    combined_pivot.plot(kind="bar", yerr=combined_err_pivot, capsize=4, ax=axes[1], rot=0)
    axes[1].set_title(f"Combined score (mean +/- std, n={benchmark_repeats})")
    axes[1].set_xlabel("Setting")
    axes[1].set_ylabel("Combined score")

    in_limits_pivot = benchmark_summary_df.pivot(index="setting", columns="method", values="in_limits_rate_mean")
    in_limits_err_pivot = benchmark_summary_df.pivot(index="setting", columns="method", values="in_limits_rate_std").fillna(0.0)
    in_limits_pivot.plot(kind="bar", yerr=in_limits_err_pivot, capsize=4, ax=axes[2], rot=0)
    axes[2].set_title(f"In-limits rate (mean +/- std, n={benchmark_repeats})")
    axes[2].set_xlabel("Setting")
    axes[2].set_ylabel("Fraction in limits")

    plt.show()

In [ ]:
# Fix aggregation so baseline (hidden_dim=None) is included in grouped results.
if 'benchmark_raw_df' in globals() and not benchmark_raw_df.empty:
    fixed_df = benchmark_raw_df.copy()
    fixed_df['hidden_dim_group'] = fixed_df['hidden_dim'].fillna(-1).astype(int)

    benchmark_summary_df = fixed_df.groupby(
        ['setting', 'method', 'epochs', 'latent_dim', 'hidden_dim_group'],
        as_index=False,
    ).agg(
        runtime_sec_mean=('runtime_sec', 'mean'),
        runtime_sec_std=('runtime_sec', 'std'),
        best_identity_mean=('best_identity', 'mean'),
        best_identity_std=('best_identity', 'std'),
        mean_identity_mean=('mean_identity', 'mean'),
        mean_identity_std=('mean_identity', 'std'),
        count_mean=('count', 'mean'),
    )

    benchmark_summary_df['hidden_dim'] = benchmark_summary_df['hidden_dim_group'].replace({-1: np.nan})
    benchmark_summary_df = benchmark_summary_df.drop(columns=['hidden_dim_group'])
    benchmark_summary_df = benchmark_summary_df.sort_values(['setting', 'method']).reset_index(drop=True)

    display(benchmark_summary_df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

    runtime_pivot = benchmark_summary_df.pivot(index='setting', columns='method', values='runtime_sec_mean')
    runtime_err_pivot = benchmark_summary_df.pivot(index='setting', columns='method', values='runtime_sec_std').fillna(0.0)
    runtime_pivot.plot(kind='bar', yerr=runtime_err_pivot, capsize=4, ax=axes[0], rot=0)
    axes[0].set_title(f'Runtime by scale setting (mean +/- std, n={benchmark_repeats})')
    axes[0].set_xlabel('Setting')
    axes[0].set_ylabel('Seconds')

    quality_pivot = benchmark_summary_df.pivot(index='setting', columns='method', values='best_identity_mean')
    quality_err_pivot = benchmark_summary_df.pivot(index='setting', columns='method', values='best_identity_std').fillna(0.0)
    quality_pivot.plot(kind='bar', yerr=quality_err_pivot, capsize=4, ax=axes[1], rot=0)
    axes[1].set_title(f'Accuracy proxy (mean best identity +/- std, n={benchmark_repeats})')
    axes[1].set_xlabel('Setting')
    axes[1].set_ylabel('Best identity to target')

    plt.show()
else:
    print('benchmark_raw_df not found. Run the benchmark cell first.')

In [ ]:
import gc
gc.collect()
if "torch" in globals() and torch.cuda.is_available():
    torch.cuda.empty_cache()